In [1]:
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q pandas numpy scikit-learn matplotlib seaborn optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 2.8 MB/s eta 0:00:00


In [2]:
# -*- coding: utf-8 -*-

# ── Imports ───────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score,
    recall_score, roc_auc_score
)

import optuna
import warnings
warnings.filterwarnings("ignore")

# ── Reproducibility ───────────────────────────────────────────────────────────
np.random.seed(42)
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# ── Dataset Configs ───────────────────────────────────────────────────────────
datasets_info = [
    {"name": "LFI", "file": "/content/drive/MyDrive/WAF_THESIS/DATASETS/lfi_cleaned.csv", "target": "class"},
    {"name": "XSS", "file": "/content/drive/MyDrive/WAF_THESIS/DATASETS/xss_cleaned.csv", "target": "class"},
    {"name": "SQL", "file": "/content/drive/MyDrive/WAF_THESIS/DATASETS/sql_cleaned.csv", "target": "class"},
    {"name": "OSC", "file": "/content/drive/MyDrive/WAF_THESIS/DATASETS/osc_cleaned.csv", "target": "class"},
]

# ── Model ─────────────────────────────────────────────────────────────────────
class TabularMLP(nn.Module):
    def __init__(self, input_dim, h1, h2, h3, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.BatchNorm1d(h1),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(h1, h2),
            nn.BatchNorm1d(h2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(h2, h3),
            nn.BatchNorm1d(h3),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(h3, 32),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)

# ── Data Preparation ──────────────────────────────────────────────────────────
def prepare_data(df, target):
    df = df.dropna()
    X = df.drop(columns=[target]).values
    y = df[target].values.astype(np.float32)

    scaler = StandardScaler()
    X = scaler.fit_transform(X).astype(np.float32)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    train_ds = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
    test_ds  = TensorDataset(torch.tensor(X_test),  torch.tensor(y_test))

    train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
    test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False)

    return X_train.shape[1], train_loader, test_loader, y_train, y_test

# ── Optuna Objective ──────────────────────────────────────────────────────────
def objective(trial, input_dim, train_loader, test_loader, y_test):

    # ── Improved Hyperparameter Search Space ──────────────────────────────────
    lr           = trial.suggest_float("lr",           1e-5, 1e-2, log=True)
    dropout      = trial.suggest_float("dropout",      0.1,  0.5)
    weight_decay = trial.suggest_float("weight_decay", 1e-7, 1e-3, log=True)
    h1           = trial.suggest_categorical("h1", [128, 256, 512])
    h2           = trial.suggest_categorical("h2", [64,  128, 256])
    h3           = trial.suggest_categorical("h3", [32,  64,  128])

    model     = TabularMLP(input_dim, h1, h2, h3, dropout).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()

    best_f1 = 0.0

    for epoch in range(40):                          # was 25, now 40
        # --- Training ---
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch).squeeze(), y_batch)
            loss.backward()
            optimizer.step()

        # --- Validation ---
        model.eval()
        preds = []
        with torch.no_grad():
            for X_batch, _ in test_loader:
                preds.extend(torch.sigmoid(model(X_batch.to(device))).cpu().numpy())

        preds     = np.array(preds).squeeze()
        preds_bin = (preds >= 0.5).astype(int)
        f1        = f1_score(y_test, preds_bin, zero_division=0)

        trial.report(f1, epoch)
        if trial.should_prune():
            raise optuna.TrialPruned()

        best_f1 = max(best_f1, f1)

    return best_f1

# ── Evaluation Helper ─────────────────────────────────────────────────────────
def evaluate(model, loader, y_true):
    model.eval()
    preds_prob = []
    with torch.no_grad():
        for X_batch, _ in loader:
            preds_prob.extend(torch.sigmoid(model(X_batch.to(device))).cpu().numpy())

    preds_prob = np.array(preds_prob).squeeze()
    preds_bin  = (preds_prob >= 0.5).astype(int)

    return {
        "Accuracy"  : accuracy_score (y_true, preds_bin),
        "F1"        : f1_score       (y_true, preds_bin, zero_division=0),
        "Precision" : precision_score(y_true, preds_bin, zero_division=0),
        "Recall"    : recall_score   (y_true, preds_bin, zero_division=0),
        "ROC-AUC"   : roc_auc_score  (y_true, preds_prob),
    }

# ── Retrain with Best Params ──────────────────────────────────────────────────
def retrain_best(best_params, input_dim, train_loader, test_loader, y_train, y_test):
    model = TabularMLP(
        input_dim,
        h1      = best_params["h1"],
        h2      = best_params["h2"],
        h3      = best_params["h3"],
        dropout = best_params["dropout"]
    ).to(device)

    optimizer = optim.Adam(
        model.parameters(),
        lr           = best_params["lr"],
        weight_decay = best_params["weight_decay"]
    )
    criterion = nn.BCEWithLogitsLoss()

    train_eval_loader = DataLoader(
        train_loader.dataset, batch_size=256, shuffle=False
    )

    EPOCHS = 100
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss = 0.0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch).squeeze(), y_batch)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        if epoch % 10 == 0:
            avg_loss = running_loss / len(train_loader)
            print(f"  Epoch {epoch:>3}/{EPOCHS}  Loss: {avg_loss:.4f}")

    print("\n  ── Train Metrics ──")
    train_metrics = evaluate(model, train_eval_loader, y_train)
    for k, v in train_metrics.items():
        print(f"    {k:<12}: {v:.4f}")

    print("\n  ── Test Metrics ───")
    test_metrics = evaluate(model, test_loader, y_test)
    for k, v in test_metrics.items():
        print(f"    {k:<12}: {v:.4f}")

    return train_metrics, test_metrics

# ── Main Loop ─────────────────────────────────────────────────────────────────
all_results = []

for ds in datasets_info:
    print("\n" + "=" * 70)
    print(f"Dataset : {ds['name']}")
    print("=" * 70)

    df = pd.read_csv(ds["file"])
    input_dim, train_loader, test_loader, y_train, y_test = prepare_data(df, ds["target"])

    # ── Optuna Study ──────────────────────────────────────────────────────────
    print("\n[Optuna] Tuning MLP hyperparameters ...")
    study = optuna.create_study(
        direction = "maximize",
        sampler   = optuna.samplers.TPESampler(seed=42),   # smarter search
        pruner    = optuna.pruners.HyperbandPruner()        # better pruning
    )
    study.optimize(
        lambda trial: objective(trial, input_dim, train_loader, test_loader, y_test),
        n_trials = 50
    )
    print(f"\n  Best Optuna F1 : {study.best_value:.4f}")
    print(f"  Best Params    : {study.best_params}")

    # ── Retrain & Evaluate ────────────────────────────────────────────────────
    print("\n[Retrain] Training with best hyperparameters ...")
    train_m, test_m = retrain_best(
        study.best_params, input_dim, train_loader, test_loader, y_train, y_test
    )

    all_results.append({
        "Dataset"   : ds["name"],
        # Test metrics only
        "Accuracy"  : round(test_m["Accuracy"],  4),
        "F1 Score"  : round(test_m["F1"],        4),
        "Precision" : round(test_m["Precision"], 4),
        "Recall"    : round(test_m["Recall"],    4),
        "ROC-AUC"   : round(test_m["ROC-AUC"],   4),
        # Best hyperparams (saved to CSV only)
        "Best LR"           : round(study.best_params["lr"],           6),
        "Best Dropout"      : round(study.best_params["dropout"],      4),
        "Best Weight Decay" : round(study.best_params["weight_decay"], 7),
        "Best H1"           : study.best_params["h1"],
        "Best H2"           : study.best_params["h2"],
        "Best H3"           : study.best_params["h3"],
    })

# ── Summary ───────────────────────────────────────────────────────────────────
results_df = pd.DataFrame(all_results)
results_df.to_csv("optuna_mlp_waf_results.csv", index=False)

print("\n" + "=" * 70)
print("FINAL RESULTS SUMMARY")
print("=" * 70)

# Clean summary table (metrics only)
summary_df = results_df[["Dataset", "Accuracy", "F1 Score", "Precision", "Recall", "ROC-AUC"]]
print(summary_df.to_string(index=False))
print("\nSaved to optuna_mlp_waf_results.csv")

Device: cpu

Dataset : LFI


[I 2026-06-15 11:58:29,468] A new study created in memory with name: no-name-7d9c41ca-88d9-4548-881f-544a59c3b12f



[Optuna] Tuning MLP hyperparameters ...


[I 2026-06-15 11:59:08,025] Trial 0 finished with value: 0.9562669071235347 and parameters: {'lr': 0.0001329291894316216, 'dropout': 0.4802857225639665, 'weight_decay': 8.471801418819978e-05, 'h1': 128, 'h2': 128, 'h3': 128}. Best is trial 0 with value: 0.9562669071235347.
[I 2026-06-15 12:00:10,559] Trial 1 finished with value: 0.9567372690401081 and parameters: {'lr': 0.00314288089084011, 'dropout': 0.18493564427131048, 'weight_decay': 5.337032762603952e-07, 'h1': 512, 'h2': 256, 'h3': 128}. Best is trial 1 with value: 0.9567372690401081.
[I 2026-06-15 12:00:12,560] Trial 2 pruned. 
[I 2026-06-15 12:00:52,582] Trial 3 finished with value: 0.9566197183098591 and parameters: {'lr': 8.200518402245828e-05, 'dropout': 0.13906884560255356, 'weight_decay': 5.456725485601477e-05, 'h1': 512, 'h2': 128, 'h3': 32}. Best is trial 1 with value: 0.9567372690401081.
[I 2026-06-15 12:00:54,194] Trial 4 pruned. 
[I 2026-06-15 12:00:56,210] Trial 5 pruned. 
[I 2026-06-15 12:01:40,230] Trial 6 finished


  Best Optuna F1 : 0.9567
  Best Params    : {'lr': 0.00314288089084011, 'dropout': 0.18493564427131048, 'weight_decay': 5.337032762603952e-07, 'h1': 512, 'h2': 256, 'h3': 128}

[Retrain] Training with best hyperparameters ...
  Epoch  10/100  Loss: 0.2023
  Epoch  20/100  Loss: 0.2002
  Epoch  30/100  Loss: 0.2002
  Epoch  40/100  Loss: 0.1987
  Epoch  50/100  Loss: 0.1996
  Epoch  60/100  Loss: 0.1998
  Epoch  70/100  Loss: 0.1985
  Epoch  80/100  Loss: 0.1981
  Epoch  90/100  Loss: 0.1982
  Epoch 100/100  Loss: 0.1987

  ── Train Metrics ──
    Accuracy    : 0.9390
    F1          : 0.9572
    Precision   : 0.9786
    Recall      : 0.9367
    ROC-AUC     : 0.9516

  ── Test Metrics ───
    Accuracy    : 0.9384
    F1          : 0.9567
    Precision   : 0.9801
    Recall      : 0.9344
    ROC-AUC     : 0.9508

Dataset : XSS


...


  Best Optuna F1 : 0.9902
  Best Params    : {'lr': 0.00314288089084011, 'dropout': 0.18493564427131048, 'weight_decay': 5.337032762603952e-07, 'h1': 512, 'h2': 256, 'h3': 128}

[Retrain] Training with best hyperparameters ...
  Epoch  10/100  Loss: 0.0325
  Epoch  20/100  Loss: 0.0279
  Epoch  30/100  Loss: 0.0237
  Epoch  40/100  Loss: 0.0257
  Epoch  50/100  Loss: 0.0215
  Epoch  60/100  Loss: 0.0202
  Epoch  70/100  Loss: 0.0238
  Epoch  80/100  Loss: 0.0212
  Epoch  90/100  Loss: 0.0189
  Epoch 100/100  Loss: 0.0186

  ── Train Metrics ──
    Accuracy    : 0.9958
    F1          : 0.9915
    Precision   : 0.9835
    Recall      : 0.9996
    ROC-AUC     : 0.9991

  ── Test Metrics ───
    Accuracy    : 0.9952
    F1          : 0.9902
    Precision   : 0.9821
    Recall      : 0.9985
    ROC-AUC     : 0.9988

Dataset : OSC


[I 2026-06-15 12:36:05,913] A new study created in memory with name: no-name-b32457c6-2a2d-4ead-9100-72ef3bdecad2



[Optuna] Tuning MLP hyperparameters ...


[I 2026-06-15 12:36:25,752] Trial 0 finished with value: 0.9864768683274021 and parameters: {'lr': 0.0001329291894316216, 'dropout': 0.4802857225639665, 'weight_decay': 8.471801418819978e-05, 'h1': 128, 'h2': 128, 'h3': 128}. Best is trial 0 with value: 0.9864768683274021.
[I 2026-06-15 12:36:59,891] Trial 1 finished with value: 0.9876835622927522 and parameters: {'lr': 0.00314288089084011, 'dropout': 0.18493564427131048, 'weight_decay': 5.337032762603952e-07, 'h1': 512, 'h2': 256, 'h3': 128}. Best is trial 1 with value: 0.9876835622927522.
[I 2026-06-15 12:37:19,650] Trial 2 finished with value: 0.9864832819539957 and parameters: {'lr': 0.00023345864076016249, 'dropout': 0.41407038455720546, 'weight_decay': 6.290644294586138e-07, 'h1': 256, 'h2': 64, 'h3': 64}. Best is trial 1 with value: 0.9876835622927522.
[I 2026-06-15 12:37:21,913] Trial 3 pruned. 
[I 2026-06-15 12:37:43,330] Trial 4 finished with value: 0.9876835622927522 and parameters: {'lr': 0.00043664735929796326, 'dropout': 


  Best Optuna F1 : 0.9877
  Best Params    : {'lr': 0.00314288089084011, 'dropout': 0.18493564427131048, 'weight_decay': 5.337032762603952e-07, 'h1': 512, 'h2': 256, 'h3': 128}

[Retrain] Training with best hyperparameters ...
  Epoch  10/100  Loss: 0.0488
  Epoch  20/100  Loss: 0.0482
  Epoch  30/100  Loss: 0.0478
  Epoch  40/100  Loss: 0.0480
  Epoch  50/100  Loss: 0.0475
  Epoch  60/100  Loss: 0.0475
  Epoch  70/100  Loss: 0.0470
  Epoch  80/100  Loss: 0.0471
  Epoch  90/100  Loss: 0.0474
  Epoch 100/100  Loss: 0.0471

  ── Train Metrics ──
    Accuracy    : 0.9851
    F1          : 0.9866
    Precision   : 0.9975
    Recall      : 0.9759
    ROC-AUC     : 0.9962

  ── Test Metrics ───
    Accuracy    : 0.9856
    F1          : 0.9870
    Precision   : 0.9981
    Recall      : 0.9761
    ROC-AUC     : 0.9965

FINAL RESULTS SUMMARY
Dataset  Accuracy  F1 Score  Precision  Recall  ROC-AUC
    LFI    0.9384    0.9567     0.9801  0.9344   0.9508
    XSS    0.9921    0.9940     0.9907  0